# NFL Player Contact Detection: Tracking Feature Model

This notebook improves on the distance-only baseline with an offline-safe tabular model. It learns from player tracking features for both player-player and player-ground rows, tunes a probability threshold with MCC, and writes `submission.csv`.


## 1. Setup and Configuration

The model is intentionally Kaggle-safe: no internet, no external packages, grouped validation by `game_play`, controlled negative sampling, and a single submission output.


In [ ]:
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

RANDOM_STATE = 42
TARGET = "contact"
ID_COL = "contact_id"
RUN_FAST = False
FAST_SAMPLE_PLAYS = 35
NEGATIVE_TO_POSITIVE_RATIO = 6
MAX_TRAIN_ROWS = 650_000
VALID_SIZE = 0.25
THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.01), 2)

REQUIRED_DATA_FILES = [
    "train_labels.csv",
    "sample_submission.csv",
    "train_player_tracking.csv",
    "test_player_tracking.csv",
]

DATA_DIR = Path("/kaggle/input/competitions/nfl-player-contact-detection")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

missing_files = [
    filename for filename in REQUIRED_DATA_FILES
    if not (DATA_DIR / filename).exists()
]
if missing_files:
    raise FileNotFoundError(
        f"Missing required files in {DATA_DIR}: {missing_files}"
    )

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")


## 2. Helper Functions

Feature construction now includes prior-step tracking dynamics, cyclic direction/orientation encodings, pair distance, pair-distance change, relative motion, angular differences, and same-team flags. These features give the model contact-onset signals that the static distance baseline cannot see.


In [ ]:
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast dataframe columns to reduce notebook memory usage."""
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5 and col != ID_COL:
                out[col] = out[col].astype("category")
    return out


def parse_contact_id(df: pd.DataFrame) -> pd.DataFrame:
    """Split Kaggle contact_id into play, step, player1, and player2 fields."""
    out = df.copy()
    parts = out[ID_COL].astype(str).str.split("_", expand=True)
    out["game_play"] = parts[0] + "_" + parts[1]
    out["step"] = parts[2].astype("int16")
    out["nfl_player_id_1"] = parts[3].astype(str)
    out["nfl_player_id_2"] = parts[4].astype(str)
    out["is_ground"] = out["nfl_player_id_2"].eq("G").astype("int8")
    out["contact_type"] = np.where(out["is_ground"].eq(1), "ground", "player_player")
    return out


def build_category_maps(*tracking_frames: pd.DataFrame) -> dict[str, dict[str, int]]:
    """Build stable integer maps for low-cardinality tracking categories."""
    maps = {}
    for col in ["position", "team"]:
        values = []
        for frame in tracking_frames:
            if col in frame.columns:
                values.extend(frame[col].astype(str).fillna("missing").unique().tolist())
        maps[col] = {value: idx for idx, value in enumerate(sorted(set(values)))}
    return maps


def prepare_tracking(tracking: pd.DataFrame, category_maps: dict[str, dict[str, int]]) -> pd.DataFrame:
    """Select, encode, and add prior-step tracking dynamics."""
    keep_cols = [
        "game_play", "step", "nfl_player_id", "x_position", "y_position",
        "speed", "distance", "direction", "orientation", "acceleration", "sa",
        "position", "team", "jersey_number",
    ]
    keep_cols = [col for col in keep_cols if col in tracking.columns]
    out = tracking[keep_cols].copy()
    out["nfl_player_id"] = out["nfl_player_id"].astype(str)
    out = out.sort_values(["game_play", "nfl_player_id", "step"]).reset_index(drop=True)

    for angle_col in ["direction", "orientation"]:
        if angle_col in out.columns:
            radians = np.deg2rad(out[angle_col])
            out[f"{angle_col}_sin"] = np.sin(radians).astype("float32")
            out[f"{angle_col}_cos"] = np.cos(radians).astype("float32")

    group = out.groupby(["game_play", "nfl_player_id"], observed=True)
    for col in ["x_position", "y_position", "speed", "distance", "acceleration", "sa"]:
        if col in out.columns:
            lag_col = f"{col}_lag1"
            out[lag_col] = group[col].shift(1)
            out[f"{col}_delta1"] = out[col] - out[lag_col]

    for col in ["direction", "orientation"]:
        if col in out.columns:
            lag = group[col].shift(1)
            diff = (out[col] - lag).abs() % 360
            out[f"{col}_delta1"] = np.minimum(diff, 360 - diff)

    for col, mapping in category_maps.items():
        if col in out.columns:
            out[f"{col}_code"] = (
                out[col].astype(str).fillna("missing").map(mapping).fillna(-1).astype("int16")
            )
        else:
            out[f"{col}_code"] = -1
    drop_cols = [col for col in ["position", "team"] if col in out.columns]
    return out.drop(columns=drop_cols)


def angular_difference(left: pd.Series, right: pd.Series) -> pd.Series:
    """Return smallest absolute angular difference in degrees."""
    diff = (left - right).abs() % 360
    return np.minimum(diff, 360 - diff)


def attach_tracking_features(contacts: pd.DataFrame, tracking: pd.DataFrame) -> pd.DataFrame:
    """Attach player tracking features and pairwise relative features."""
    base = contacts.copy()
    track = tracking.copy()
    player_cols = [
        col for col in track.columns
        if col not in ["game_play", "step", "nfl_player_id"]
    ]

    out = base.merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_1"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).drop(columns=["nfl_player_id"])
    out = out.rename(columns={col: f"p1_{col}" for col in player_cols})

    p2_rows = base[base["is_ground"].eq(0)].copy()
    p2 = p2_rows[[ID_COL, "game_play", "step", "nfl_player_id_2"]].merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_2"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).drop(columns=["game_play", "step", "nfl_player_id_2", "nfl_player_id"])
    p2 = p2.rename(columns={col: f"p2_{col}" for col in player_cols})
    out = out.merge(p2, on=ID_COL, how="left")

    out["dx"] = out["p1_x_position"] - out["p2_x_position"]
    out["dy"] = out["p1_y_position"] - out["p2_y_position"]
    out["pair_distance"] = np.sqrt(np.square(out["dx"]) + np.square(out["dy"]))
    out["abs_dx"] = out["dx"].abs()
    out["abs_dy"] = out["dy"].abs()

    lag_cols = [
        "p1_x_position_lag1", "p2_x_position_lag1",
        "p1_y_position_lag1", "p2_y_position_lag1",
    ]
    if all(col in out.columns for col in lag_cols):
        out["dx_lag1"] = out["p1_x_position_lag1"] - out["p2_x_position_lag1"]
        out["dy_lag1"] = out["p1_y_position_lag1"] - out["p2_y_position_lag1"]
        out["pair_distance_lag1"] = np.sqrt(
            np.square(out["dx_lag1"]) + np.square(out["dy_lag1"])
        )
        out["pair_distance_delta1"] = out["pair_distance"] - out["pair_distance_lag1"]
        out["abs_pair_distance_delta1"] = out["pair_distance_delta1"].abs()

    for col in ["speed", "distance", "acceleration", "sa"]:
        left = f"p1_{col}"
        right = f"p2_{col}"
        if left in out.columns and right in out.columns:
            out[f"{col}_diff"] = out[left] - out[right]
            out[f"abs_{col}_diff"] = out[f"{col}_diff"].abs()

    for col in ["direction", "orientation"]:
        left = f"p1_{col}"
        right = f"p2_{col}"
        if left in out.columns and right in out.columns:
            out[f"{col}_angle_diff"] = angular_difference(out[left], out[right])

    if "p1_team_code" in out.columns and "p2_team_code" in out.columns:
        out["same_team"] = (out["p1_team_code"] == out["p2_team_code"]).astype("float32")

    out["step_float"] = out["step"].astype("float32")
    return out


def sample_training_rows(labels: pd.DataFrame) -> pd.DataFrame:
    """Keep all positives and a controlled number of negatives for training."""
    if RUN_FAST:
        labels = labels[labels["game_play"].isin(
            pd.Series(labels["game_play"].unique()).sample(
                min(FAST_SAMPLE_PLAYS, labels["game_play"].nunique()),
                random_state=RANDOM_STATE,
            )
        )].copy()

    positives = labels[labels[TARGET].eq(1)]
    negatives = labels[labels[TARGET].eq(0)]
    max_negatives = min(
        len(negatives),
        max(len(positives) * NEGATIVE_TO_POSITIVE_RATIO, MAX_TRAIN_ROWS - len(positives)),
    )
    negatives = negatives.sample(max_negatives, random_state=RANDOM_STATE)
    sampled = pd.concat([positives, negatives], ignore_index=True)
    if len(sampled) > MAX_TRAIN_ROWS:
        sampled = sampled.sample(MAX_TRAIN_ROWS, random_state=RANDOM_STATE)
    return sampled.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)


def get_feature_columns(df: pd.DataFrame) -> list[str]:
    """Return numeric model features, excluding IDs and target columns."""
    excluded = {
        ID_COL, TARGET, "game_play", "nfl_player_id_1", "nfl_player_id_2",
        "contact_type", "datetime",
    }
    return [
        col for col in df.columns
        if col not in excluded and pd.api.types.is_numeric_dtype(df[col])
    ]


def score_thresholds(y_true: np.ndarray, probabilities: np.ndarray) -> pd.DataFrame:
    """Score probability thresholds with MCC and positive rate."""
    rows = []
    for threshold in THRESHOLDS:
        pred = (probabilities >= threshold).astype("int8")
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, pred, average="binary", zero_division=0
        )
        rows.append({
            "threshold": threshold,
            "mcc": matthews_corrcoef(y_true, pred),
            "positive_rate": pred.mean(),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })
    return pd.DataFrame(rows).sort_values("mcc", ascending=False)


def make_model():
    """Create the strongest offline-safe sklearn model available."""
    try:
        return HistGradientBoostingClassifier(
            learning_rate=0.06,
            max_iter=240,
            max_leaf_nodes=31,
            l2_regularization=0.05,
            early_stopping=True,
            validation_fraction=0.12,
            random_state=RANDOM_STATE,
        )
    except TypeError:
        return RandomForestClassifier(
            n_estimators=240,
            min_samples_leaf=8,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )


## 3. Load Data


In [ ]:
labels = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_labels.csv",
    parse_dates=["datetime"],
))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_player_tracking.csv",
    parse_dates=["datetime"],
))
test_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "test_player_tracking.csv",
    parse_dates=["datetime"],
))

labels = parse_contact_id(labels)
sample_submission = parse_contact_id(sample_submission)
category_maps = build_category_maps(train_tracking, test_tracking)
train_tracking_prepared = prepare_tracking(train_tracking, category_maps)
test_tracking_prepared = prepare_tracking(test_tracking, category_maps)

print(f"labels: {labels.shape}")
print(f"sample_submission: {sample_submission.shape}")
print(f"train_tracking: {train_tracking_prepared.shape}")
print(f"test_tracking: {test_tracking_prepared.shape}")
print(f"positive rate: {labels[TARGET].mean():.5f}")
labels.head()


## 4. Grouped Split and Feature Construction

Split by `game_play` before downsampling. The model trains on all positives plus sampled negatives from the training plays, while validation keeps the natural held-out play distribution for realistic MCC threshold tuning.


In [ ]:

model_source = labels.copy()
if RUN_FAST:
    model_source = model_source[model_source["game_play"].isin(
        pd.Series(model_source["game_play"].unique()).sample(
            min(FAST_SAMPLE_PLAYS, model_source["game_play"].nunique()),
            random_state=RANDOM_STATE,
        )
    )].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
)
train_source_idx, valid_source_idx = next(
    splitter.split(model_source, model_source[TARGET], groups=model_source["game_play"])
)
train_source = model_source.iloc[train_source_idx].copy()
valid_rows = model_source.iloc[valid_source_idx].copy()
train_rows = sample_training_rows(train_source)

print(f"source rows: {model_source.shape}")
print(f"training-source rows: {train_source.shape}")
print(f"sampled training rows: {train_rows.shape}")
print(f"natural validation rows: {valid_rows.shape}")
print(f"sampled training positive rate: {train_rows[TARGET].mean():.5f}")
print(f"natural validation positive rate: {valid_rows[TARGET].mean():.5f}")
display(train_rows.groupby("contact_type", observed=True)[TARGET].agg(["count", "mean", "sum"]))
display(valid_rows.groupby("contact_type", observed=True)[TARGET].agg(["count", "mean", "sum"]))

train_features = attach_tracking_features(train_rows, train_tracking_prepared)
valid_features = attach_tracking_features(valid_rows, train_tracking_prepared)
feature_cols = get_feature_columns(train_features)

print(f"train feature frame: {train_features.shape}")
print(f"valid feature frame: {valid_features.shape}")
print(f"feature count: {len(feature_cols)}")
display(train_features[feature_cols].head())

del model_source, train_source, train_rows, valid_rows
gc.collect()


## 5. Train/Validation Matrices

Prepare numeric matrices for the model. Missing player-2 values are expected for ground-contact rows and are handled by histogram gradient boosting.


In [ ]:

X_train = train_features[feature_cols].replace([np.inf, -np.inf], np.nan)
y_train = train_features[TARGET].astype("int8")
X_valid = valid_features[feature_cols].replace([np.inf, -np.inf], np.nan)
y_valid = valid_features[TARGET].astype("int8")
valid_meta = valid_features[[ID_COL, "game_play", "contact_type"]].copy()

print(f"train rows: {len(X_train):,}")
print(f"valid rows: {len(X_valid):,}")
print(f"train positive rate: {y_train.mean():.5f}")
print(f"valid positive rate: {y_valid.mean():.5f}")


## 6. Train Model and Tune Threshold

The model outputs probabilities. MCC is optimized by searching hard-label thresholds on held-out plays.


In [ ]:
model = make_model()
model.fit(X_train, y_train)
valid_prob = model.predict_proba(X_valid)[:, 1]
threshold_scores = score_thresholds(y_valid.to_numpy(), valid_prob)
best_threshold = float(threshold_scores.iloc[0]["threshold"])

print(f"model: {type(model).__name__}")
print(f"best threshold: {best_threshold:.2f}")
display(threshold_scores.head(12))

valid_pred = (valid_prob >= best_threshold).astype("int8")
valid_scored = valid_meta.assign(
    y_true=y_valid.to_numpy(),
    probability=valid_prob,
    prediction=valid_pred,
)
slice_scores = []
for contact_type, part in valid_scored.groupby("contact_type", observed=True):
    slice_scores.append({
        "contact_type": contact_type,
        "rows": len(part),
        "actual_rate": part["y_true"].mean(),
        "predicted_rate": part["prediction"].mean(),
        "mcc": matthews_corrcoef(part["y_true"], part["prediction"]),
    })
display(pd.DataFrame(slice_scores))


## 7. Train Final Model and Create Submission

Refit on the sampled training rows, score every submission row, and write `submission.csv`.


In [ ]:

final_train_rows = sample_training_rows(labels)
final_features = attach_tracking_features(final_train_rows, train_tracking_prepared)
X_full = final_features[feature_cols].replace([np.inf, -np.inf], np.nan)
y_full = final_features[TARGET].astype("int8")
final_model = make_model()
final_model.fit(X_full, y_full)

submission_rows = sample_submission[[
    ID_COL, "game_play", "step", "nfl_player_id_1", "nfl_player_id_2",
    "is_ground", "contact_type",
]].copy()
test_features = attach_tracking_features(submission_rows, test_tracking_prepared)
X_test = test_features[feature_cols].replace([np.inf, -np.inf], np.nan)
test_prob = final_model.predict_proba(X_test)[:, 1]

submission = sample_submission[[ID_COL]].copy()
submission[TARGET] = (test_prob >= best_threshold).astype("int8")
output_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(output_path, index=False)

print(f"wrote: {output_path}")
print(f"submission shape: {submission.shape}")
print(f"predicted positive rate: {submission[TARGET].mean():.5f}")
display(submission.head())

del final_train_rows, final_features, X_full, X_test, test_features
gc.collect()


## 8. Next Improvements

This model should beat the pure distance baseline because it can predict ground rows and use movement dynamics. Next upgrades should add nearest-player context, helmet box visibility/size features, video-derived posture cues, and post-threshold smoothing validated by play-grouped OOF predictions.
